In [1]:
import pandas as pd
gares = pd.read_csv("../../data/raw/gares.csv", sep=";", dtype=str)
gares.head()
gares.columns

Index(['Nom_Gare', 'Trigramme', 'Segment(s) DRG', 'Position géographique',
       'Code commune', 'Code_UIC', 'Id_Gare'],
      dtype='str')

In [3]:
st = gares[["Id_Gare", "Nom_Gare", "Code_UIC", "Position géographique", "Code commune"]].copy()
st.columns = ["station_id", "name", "uic", "position", "insee"]

In [4]:
st[["lat", "lon"]] = st["position"].str.split(",", expand=True).astype(float)
st[st["lat"].isna()] 

,station_id,name,uic,position,insee,lat,lon


In [7]:
# print(st.head())
for i in range(len(st)):
    if st.loc[i, "lat"] is None:
        print("Station without coordinates:", st.loc[i, "name"], st.loc[i, "position"])
    elif st.loc[i, "lat"] < 40 or st.loc[i, "lat"] > 52:
        print("Station with invalid latitude:", st.loc[i, "name"], st.loc[i, "lat"])
    elif st.loc[i, "lon"] < -5 or st.loc[i, "lon"] > 10:
        print("Station with invalid longitude:", st.loc[i, "name"], st.loc[i, "lon"])


In [20]:
st[st["lat"].isna() | st["lon"].isna()]                    
st[~st["lat"].between(41, 52) | ~st["lon"].between(-5.5, 10)]  

,station_id,name,uic,position,insee,lat,lon


In [9]:
freq = pd.read_csv("../../data/raw/frequentation_latest.csv", sep=";", dtype=str)
freq.columns

Index(['Nom de la gare', 'Code UIC', 'Code postal',
       'Direction Régionale Gares', 'Segmentation DRG',
       'Segmentation Marketing', 'Total Voyageurs 2025',
       'Total Voyageurs + Non voyageurs 2025'],
      dtype='str')

In [19]:
print(st["uic"].head())
for i in range(len(st)):
    if len(st["uic"].iloc[i]) != 8:
        print("UIC code with invalid length:", st["uic"].iloc[i], st["name"].iloc[i])

0    87313759
1    87481614
2    87317362
3    87545269
4    87386052
Name: uic, dtype: str
UIC code with invalid length: 87001479;87271494 Aéroport Charles de Gaulle 2 TGV
UIC code with invalid length: 87318964;87981902 Avignon TGV
UIC code with invalid length: 87590372;87481648 Châteaubriant
UIC code with invalid length: 87534131;87276055 Ermont - Eaubonne
UIC code with invalid length: 87733667;87382812 Mareil-Marly
UIC code with invalid length: 87393876;87733659 Noisy-le-Roi
UIC code with invalid length: 87547026;87547000 Paris Austerlitz
UIC code with invalid length: 87686006;87686030 Paris Gare de Lyon
UIC code with invalid length: 87271031;87271023;87271007 Paris Gare du Nord
UIC code with invalid length: 87391011;87391003;87391102 Paris Montparnasse
UIC code with invalid length: 87744078;87746479 Saint-Gervais-les-Bains Le Fayet


In [14]:
print(freq["Code UIC"].head())

0    87313759
1    87271460
2    87313098
3    87746115
4    87615005
Name: Code UIC, dtype: str


In [17]:
count = 0
for i in range(len(freq)):
    if freq["Code UIC"].iloc[i] not in st.uic.values:
        print("UIC code not found in stations:", freq["Code UIC"].iloc[i], freq["Nom de la gare"].iloc[i])
        count += 1

print("Total UIC codes not found in stations:", count)

UIC code not found in stations: 87318964 Avignon TGV
UIC code not found in stations: 87297291 Barenton-Bugny
UIC code not found in stations: 87144188 Biffontaine
UIC code not found in stations: 87615229 Campagne-sur-Aude
UIC code not found in stations: 87592873 Chabanais
UIC code not found in stations: 87175737 Clermont-en-Argonne
UIC code not found in stations: 87584425 Couze
UIC code not found in stations: 87413732 Épouville
UIC code not found in stations: 87276055 Ermont - Eaubonne
UIC code not found in stations: 87615211 Espéraza
UIC code not found in stations: 87316752 Feuquerolles
UIC code not found in stations: 87317503 Feuquières - Fressenneville
UIC code not found in stations: 87175075 Fronville
UIC code not found in stations: 87618082 Golfech
UIC code not found in stations: 87215145 Heming Mairie
UIC code not found in stations: 87191346 Kédange
UIC code not found in stations: 87583427 La Rochefoucauld
UIC code not found in stations: 87611129 Labarthe-Inard
UIC code not found 

In [23]:
print(st["uic"].str.len().value_counts())
print(freq["Code UIC"].str.len().value_counts())

uic
8     2781
17       9
26       2
Name: count, dtype: int64
Code UIC
8    2998
Name: count, dtype: int64


In [24]:
st_uic = st.assign(uic=st["uic"].str.split(r"[;,| ]+")).explode("uic")
print(st_uic["uic"].str.len().value_counts())
print(freq["Code UIC"].str.len().value_counts())

uic
8    2805
Name: count, dtype: int64
Code UIC
8    2998
Name: count, dtype: int64


In [30]:
missing = freq[~freq["Code UIC"].isin(st_uic["uic"])]
len(missing)

212

In [28]:
st_uic[st_uic["name"].str.contains("Avignon|Ermont", case=False)][["name", "uic"]]

,name,uic
149,Avignon Centre,87765008
150,Avignon TGV,87318964
150,Avignon TGV,87981902
151,Avignonet-Lauragais,87616011
585,Clermont de l'Oise,87313304
586,Clermont La Pardieu,87782607
587,Clermont La Rotonde,87396895
588,Clermont-Ferrand,87734004
789,Ermont - Eaubonne,87534131
789,Ermont - Eaubonne,87276055


,Nom de la gare,Code UIC,Code postal,Direction Régionale Gares,Segmentation DRG,Segmentation Marketing,Total Voyageurs 2025,Total Voyageurs + Non voyageurs 2025


In [32]:
def key(s):
    return (s.str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
             .str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip())

st_uic["key"] = key(st_uic["name"])
missing = missing.assign(key=key(missing["Nom de la gare"]))

found_by_name = missing[missing["key"].isin(st_uic["key"])]
not_in_list = missing[~missing["key"].isin(st_uic["key"])]
len(found_by_name), len(not_in_list)

(0, 212)

In [35]:
print(missing[["Nom de la gare", "key"]].head())  
print(st_uic[["name", "key"]].head())
print(st_uic[st_uic["name"].str.contains("Luchon|Marans|Kédange", case=False)])  

          Nom de la gare                  key
43        Barenton-Bugny       barenton bugny
63           Biffontaine          biffontaine
101    Campagne-sur-Aude    campagne sur aude
112            Chabanais            chabanais
135  Clermont-en-Argonne  clermont en argonne
                    name                    key
0              Abancourt              abancourt
1               Abbaretz               abbaretz
2              Abbeville              abbeville
3        Ablon-sur-Seine        ablon sur seine
4  Achères Grand Cormier  acheres grand cormier
Empty DataFrame
Columns: [station_id, name, uic, position, insee, lat, lon, key]
Index: []


In [37]:
vis = pd.to_numeric(freq["Total Voyageurs 2025"], errors="coerce")  # 実際の列名に合わせて
is_missing = freq["Code UIC"].isin(missing["Code UIC"])
vis[is_missing].sum() / vis.sum(), vis[is_missing].describe()

(np.float64(0.0033002314790724955),
 count    2.120000e+02
 mean     5.002600e+04
 std      6.206902e+05
 min      0.000000e+00
 25%      0.000000e+00
 50%      0.000000e+00
 75%      4.922500e+02
 max      8.997002e+06
 Name: Total Voyageurs 2025, dtype: float64)

In [38]:
missing.loc[vis[is_missing].idxmax()]
# または上位5駅を見る
missing.assign(vis=vis[is_missing]).nlargest(5, "vis")[["Nom de la gare", "Code UIC", "vis"]]


,Nom de la gare,Code UIC,vis
377,Monaco - Monte-Carlo,87756403,8997002.0
1089,Saint-Germain-en-Laye Bel-Air - Fourqueux,87366922,918024.0
1584,Mirecourt,87144212,76177.0
338,Luchon,87611236,74012.0
1549,Lutterbach,87182667,63801.0


In [39]:
f = freq[["Code UIC", "Total Voyageurs 2025"]].rename(
    columns={"Code UIC": "uic", "Total Voyageurs 2025": "visitors"})
f["visitors"] = pd.to_numeric(f["visitors"], errors="coerce")

merged = st_uic.merge(f, on="uic", how="left")   

stations = merged.groupby("station_id", as_index=False).agg(
    name=("name", "first"),
    uic=("uic", ";".join),                       
    lat=("lat", "first"),
    lon=("lon", "first"),
    insee=("insee", "first"),
    visitors=("visitors", lambda s: s.sum(min_count=1)),
)

In [40]:
len(stations), len(st)                     
stations["visitors"].isna().sum()       
stations.nlargest(10, "visitors")[["name", "visitors"]] 

,name,visitors
1196,Paris Gare du Nord,263160032.0
1801,Paris Gare de Lyon,122546761.0
1631,Paris Saint-Lazare,111236515.0
2410,Paris Montparnasse,70170177.0
2707,Lyon Part Dieu,47278206.0
1012,Paris Est,43352080.0
1223,Juvisy-sur-Orge,41634282.0
1933,Magenta,40348994.0
1132,Haussmann Saint-Lazare,39825373.0
2044,Saint-Denis,37300784.0
